## Start modelering van voorspellingsmodel

### Import data

In [ ]:
import pandas as pd
import category_encoders as ce
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn import metrics

df = pd.read_csv("../../csv/new_train_data/train_CarBreakDown.csv")

# Split method

# df_safe = df[df['breakdown_next_30_days'] == 0]
# df_breakdown = df[df['breakdown_next_30_days'] == 1]

### Splits data in x en y en in train en test data

In [ ]:
used_features =['vehicle_brand','vehicle_age_years','km_per_engine_hour','service_ratio','oil_quality_pct','avg_trip_length_km','weather_exposure','fuel_type','cleanliness_score','driver_satisfaction_score','tyre_type']

x = df[used_features]
y = df[['breakdown_next_30_days']]

# Split method

# x_safe = df_safe[used_features]
# x_breakdown = df_breakdown[used_features]
# y_safe = df_safe[['breakdown_next_30_days']]
# y_breakdown = df_breakdown[['breakdown_next_30_days']]

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2)


In [ ]:
x

In [ ]:
y

### Encoding van categorische velden

In [ ]:
# !pip install --upgrade category_encoders

category_encoder = ce.OrdinalEncoder(cols = ['vehicle_brand','weather_exposure','fuel_type','tyre_type'])

x_encoded = category_encoder.fit_transform(x_train)

# Split method

# x_safe_encoded = category_encoder.fit_transform(x_safe)
# x_breakdown_encoded = category_encoder.transform(x_breakdown)
x_test_encoded = category_encoder.transform(x_test)
x_encoded

from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
x_encoded, y_train = smote.fit_resample(x_encoded, y_train)

### Train model via Random forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from random import randint

# Split method

# end_df = pd.DataFrame([])
# end_df = pd.concat([end_df, x_breakdown_encoded])
# end_df = pd.concat([end_df, x_safe_encoded.sample(n=x_breakdown_encoded.shape[0])])

clf = RandomForestClassifier(criterion='entropy',n_estimators=100, max_samples = 50, class_weight="balanced")
clf = clf.fit(x_encoded, y_train.values.ravel()) # values.ravel() because a warning appears about using a 1d array so we need to flatten it.




from xgboost import XGBClassifier
# create model
model = XGBClassifier(
    n_estimators=500,
    learning_rate=0.01,
    max_depth=6,
    random_state=42
)

# train
model.fit(x_encoded, y_train)

# Split method

# use end_df and .sample(n=x_breakdown_encoded.shape[0]*2) instead for the fit



# proba = model.predict_proba(x_test_encoded)[:,0]

# pred = (proba > 0.95).astype(int)
# pred

#### Voorspelling output

In [ ]:
clf_prediction = clf.predict(x_test_encoded)
print(clf_prediction)

#### Accuracy weergave

In [ ]:
print(f"Accuracy: {metrics.accuracy_score(y_test, clf_prediction)}")

#### Confusion matrix

In [ ]:
confusion_matrix(y_test, clf_prediction)

In [ ]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(model, x_encoded, y_train, cv=5, scoring="f1")
print("Mean F1:", scores.mean())

In [ ]:
from xgboost import plot_importance
import matplotlib.pyplot as plt

plot_importance(model)
plt.show()

## Spliting the break down and safe results

One of the problems in the dataset was that there is a considarably high amount of non breaking cars. To try and solve this we split the data in two data frames (see # split method), one with only working cars and one with all the cars that brokedown. Then when fitting the model we took a sample of 100 from both then dataframes and combined them to form the train data. This had little effect on the end result with the accurasy and false positives not changing.

In addition we also tried to use duplicate rows of the breakdwon dataframe to see if the problem was the lack of data but this also dit not change it.

## changing prediction certainty treshholds
proba = clf.predict_proba(x_test_encoded)[:,0]

pred = (proba > 0.81).astype(int)

pred

This code makes a model which requires predictions to be of a certain certainty. This solved the issue of always guessing 0 but also tanked the accuracy.